# Caption student — QLoRA fine-tune (Qwen3-VL-2B ← Qwen3-VL-235B teacher)

Caption-only distillation. Trains a LoRA adapter so a small **Qwen3-VL-2B** describes an
octopus clip in one sentence, mimicking the 235B teacher captions.

**Before running:** locally run `python3 build_caption_dataset.py`, then upload the produced
`src/dataset/vN.zip` to `MyDrive/GSOC-Catrobat/caption-student/` (set `DATASET_ZIP` below).

**Runtime → A100 GPU.** Run cells top to bottom. The **Phase-0 smoke test** proves Qwen3-VL
trains in 4-bit+LoRA before the full run — if it errors, set `MODEL_ID` to the Qwen2.5-VL-3B
fallback and re-run from the model-setup cell (nothing else changes).

## 1. Install

In [ ]:
# Qwen3-VL needs a recent transformers. If AutoModelForImageTextToText can't load Qwen3-VL,
# swap the pinned line for:  !pip -q install -U git+https://github.com/huggingface/transformers
!pip -q install -U "transformers>=4.57.0" accelerate peft bitsandbytes qwen-vl-utils sentence-transformers rouge-score
# PIN pillow: `-U pillow` pulls 11.3.0 whose ImageText->_Ink import breaks Colab's torchvision. 11.2.1 is clean.
!pip -q install "pillow==11.2.1"
!pip -q uninstall -y torchao 2>/dev/null   # torchao clashes with recent transformers (restart runtime if prompted)
import torch; print("torch", torch.__version__, "| gpu", torch.cuda.get_device_name(0))

## 2. Config

In [ ]:
from pathlib import Path
import json
from google.colab import drive; drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/GSOC-Catrobat")
WORK  = DRIVE / "caption-student"; WORK.mkdir(parents=True, exist_ok=True)

MODEL_ID    = "Qwen/Qwen3-VL-2B-Instruct"      # FALLBACK: "Qwen/Qwen2.5-VL-3B-Instruct"
VERSION_TAG = "v1"                             # dataset snapshot version (matches build_caption_dataset.py)
DATASET_ZIP = WORK / f"{VERSION_TAG}.zip"      # upload the snapshot zip here
DATASET_DIR = Path("/content/dataset")         # local unzip target (fast disk)
OUT_DIR     = WORK / "lora_out"; OUT_DIR.mkdir(exist_ok=True)

# LoRA + training knobs
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LR, EPOCHS, GRAD_ACCUM = 1e-4, 3, 16
PROMPT = ("These frames are from one short aquarium clip of Nity, an octopus, in time order. "
          "Describe in ONE sentence what the octopus is doing.")
assert DATASET_ZIP.exists(), f"upload the dataset snapshot to {DATASET_ZIP}" 

## 3. Unpack dataset snapshot

In [ ]:
import zipfile
DATASET_DIR.mkdir(exist_ok=True)
if (DATASET_DIR/"train.jsonl").exists():
    print("dataset already extracted — skipping unzip")
else:
    with zipfile.ZipFile(DATASET_ZIP) as z: z.extractall(DATASET_DIR)
train = [json.loads(l) for l in open(DATASET_DIR/"train.jsonl")]
val   = [json.loads(l) for l in open(DATASET_DIR/"val.jsonl")]
print("train", len(train), "| val", len(val))
print("sample caption:", train[0]["caption"])
print("sample frames :", len(train[0]["frames"]))

## 4. Model + LoRA (4-bit)

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from qwen_vl_utils import process_vision_info

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
# min/max_pixels bound image-token count (memory). ~256..1280 * 28^2.
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True,
        min_pixels=256*28*28, max_pixels=1024*28*28)
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, quantization_config=bnb,
        torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
model = prepare_model_for_kbit_training(model)
lora = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 5. Example builder (chat format + label masking)

In [ ]:
from torch.utils.data import Dataset

def build_example(rec, with_target=True):
    """Format one clip -> model inputs. Loss is supervised ONLY on the caption tokens
    (prompt + image tokens masked to -100)."""
    frame_paths = [str(DATASET_DIR / f) for f in rec["frames"]]
    user = [{"type": "image", "image": p} for p in frame_paths] + [{"type": "text", "text": PROMPT}]
    msgs = [{"role": "user", "content": user}]
    if with_target:
        msgs.append({"role": "assistant", "content": [{"type": "text", "text": rec["caption"]}]})
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=not with_target)
    imgs, vids = process_vision_info(msgs)
    enc = processor(text=[text], images=imgs, videos=vids, return_tensors="pt", padding=True)
    if with_target:
        ptext = processor.apply_chat_template([{"role": "user", "content": user}],
                                              tokenize=False, add_generation_prompt=True)
        penc = processor(text=[ptext], images=imgs, videos=vids, return_tensors="pt", padding=True)
        plen = penc["input_ids"].shape[1]                 # image tokens already expanded -> prefix of full seq
        labels = enc["input_ids"].clone(); labels[:, :plen] = -100
        enc["labels"] = labels
    return {k: v for k, v in enc.items()}

class CapDS(Dataset):
    def __init__(self, recs): self.recs = recs
    def __len__(self): return len(self.recs)
    def __getitem__(self, i): return self.recs[i]

def collate(batch): return build_example(batch[0], with_target=True)   # batch size 1 (grad-accum for effective batch)

## 6. PHASE 0 — smoke test (one forward+backward)

Proves the arch trains in 4-bit+LoRA before the full run. If this errors, switch MODEL_ID to the Qwen2.5-VL-3B fallback and re-run from cell 4.

In [ ]:
ex = build_example(train[0]); ex = {k: v.to(model.device) for k, v in ex.items()}
model.train()
out = model(**ex)
print("loss:", float(out.loss))
out.loss.backward()
model.zero_grad()
print("SMOKE OK — Qwen3-VL trains in 4-bit + LoRA")

## 7. Train

In [ ]:
from transformers import TrainingArguments, Trainer
import os, glob

CKPT_DIR = str(OUT_DIR / f"qwen3vl2b_caption_{VERSION_TAG}_ckpt")   # on Drive -> survives runtime disconnect

model.config.use_cache = False
targs = TrainingArguments(
    output_dir=CKPT_DIR,
    per_device_train_batch_size=1, gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR, num_train_epochs=EPOCHS, warmup_ratio=0.03, lr_scheduler_type="cosine",
    bf16=True, gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=10,
    save_strategy="steps", save_steps=50, save_total_limit=2,   # checkpoint to Drive (~25min) so a disconnect is recoverable
    remove_unused_columns=False, report_to="none")
trainer = Trainer(model=model, args=targs, train_dataset=CapDS(train), data_collator=collate)

# AUTO-RESUME: if a checkpoint from a previous (disconnected) run exists on Drive, continue from it.
_ckpts = glob.glob(os.path.join(CKPT_DIR, "checkpoint-*"))
if _ckpts:
    last = max(_ckpts, key=lambda p: int(p.split("-")[-1]))
    print(f"RESUMING from {last}", flush=True)
    trainer.train(resume_from_checkpoint=last)
else:
    print("fresh start (no prior checkpoint on Drive)", flush=True)
    trainer.train()

## 8. Save adapter to Drive

In [ ]:
adir = OUT_DIR / f"qwen3vl2b_caption_{VERSION_TAG}"
model.save_pretrained(adir); processor.save_pretrained(adir)
json.dump({"model_id": MODEL_ID, "dataset_version": VERSION_TAG,
           "n_train": len(train), "n_val": len(val), "lora_r": LORA_R, "epochs": EPOCHS},
          open(adir/"train_meta.json", "w"), indent=2)
!cd "{OUT_DIR}" && zip -qr "{adir.name}.zip" "{adir.name}"
print("saved ->", adir, "(+ .zip)")

## 9. Eval — base vs LoRA on the held-out val split

In [ ]:
import contextlib, numpy as np
from sentence_transformers import SentenceTransformer, util
from rouge_score import rouge_scorer

st = SentenceTransformer("all-MiniLM-L6-v2")
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
model.config.use_cache = True; model.eval()

def generate(rec, use_lora=True):
    enc = build_example(rec, with_target=False); enc = {k: v.to(model.device) for k, v in enc.items()}
    ctx = contextlib.nullcontext() if use_lora else model.disable_adapter()
    with torch.no_grad(), ctx:
        out = model.generate(**enc, max_new_tokens=64, do_sample=False)
    return processor.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

N = min(50, len(val)); rows = []
for rec in val[:N]:
    rows.append((rec["caption"], generate(rec, use_lora=False), generate(rec, use_lora=True)))

def sim(a, b):
    e = st.encode([a, b], convert_to_tensor=True); return float(util.cos_sim(e[0], e[1]))
for name, idx in [("BASE", 1), ("LoRA", 2)]:
    s  = np.mean([sim(r[0], r[idx]) for r in rows])
    rl = np.mean([scorer.score(r[0], r[idx])["rougeL"].fmeasure for r in rows])
    print(f"{name}: emb-sim={s:.3f}  rougeL={rl:.3f}")
print("-"*70)
for r in rows[:8]:
    print("REF :", r[0]); print("BASE:", r[1]); print("LoRA:", r[2]); print()